Let's do a little example similar to the Cannon problem.
Instead of spectra we can use time-series from finance.
We'll set up a small problem where we want to predict the values
of a stock (like the "flux") using the labels from other
two other stocks.
(Here we are working in time rather than wavelength.)
(Note: this is not a real predictive model for the stock market...do not use this!)

In [ ]:
#!pip install -U yfinance==0.2.55 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
import seaborn as sns

import yfinance as yf

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

Grab some data. Let's use Nike and Chipotle values as the labels and we'll see if we can come up with a model to predict Google.

In [ ]:
labels = ["NKE", "CMG"]
google = "GOOG"
df = yf.download(labels, start="2025-01-01", end="2025-3-20")
df_google = yf.download(google, start="2025-01-01", end="2025-3-20")

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))

# Top panel: label stocks (Nike, Chipotle)
for label in labels:
    ax1.plot(df.index, df[("Open", label)] / df[("Open", label)].median(),
             alpha=0.8, label=label)
ax1.set_ylabel("Normalized Open Price")
ax1.legend()
ax1.set_title("Label Stocks")

# Bottom panel: target stock (Google)
ax2.plot(df_google.index, df_google["Open"] / df_google["Open"].median(),
         linewidth=3, alpha=0.8, label=google, color=sns.color_palette()[2])
ax2.set_ylabel("Normalized Open Price")
ax2.legend()
ax2.set_title("Target Stock")

fig.tight_layout()

Let predict the Google using the price of the other stocks. We'll assume 
4 different timeseries (like 4 different "stars") here: open, close, high, low.

${\rm Google}_{\rm open}(t) = \theta_{0, t} 
    + \theta_{1, t} {\rm Nike}_{\rm open}(t) +  \theta_{2, t}{\rm Chipotle}_{\rm open}(t) \ldots $

${\rm Google}_{\rm close}(t) = \theta_{0, t} 
    + \theta_{1, t} {\rm Nike}_{\rm close}(t) +  \theta_{2, t}{\rm Chipotle}_{\rm close}(t) \ldots $


${\rm Google}_{\rm high}(t) = \theta_{0, t} + \ldots $

$\vec y_t = {\mathbf{X}_t} \vec \theta_t$

with 

$${\mathbf{X}_t} = \begin{pmatrix}
1 &  {\rm Nike}_{\rm open}(t) & 
    {\rm Chipotle}_{\rm open} & \ldots \\
1 & {\rm Nike}_{\rm close}(t) & 
    {\rm Chipotle}_{\rm close} & \ldots\\
1 &  {\rm Nike}_{\rm high}(t) & 
    {\rm Chipotle}_{\rm high}& \ldots \\
1 &  {\rm Nike}_{\rm low}(t) & 
    {\rm Chipotle}_{\rm low} & \ldots
\end{pmatrix}
$$

and

$$\theta_t = \begin{pmatrix}
\theta_{0, t}\\
\theta_{1, t}\\
\theta_{2, t}
\end{pmatrix}
$$

Let's look at the data

In [ ]:
label_df = df.iloc[:][["Open", "Close", "High", "Low"]]
google_df = df_google.iloc[:][["Open", "Close", "High", "Low"]]
label_df

Now let's pick a time to predict. Again we're trying to predict 4 different values of Google stock using the labels from others stocks.

In [ ]:
t = 0
label_time_step_t = label_df.iloc[t].values.reshape(4, len(labels))
google_time_step_t = google_df.iloc[t].values.reshape(4, 1)

In [ ]:
label_time_step_t

In [ ]:
google_time_step_t

Set up the ${\mathbf{X}}$ matrix at this timestep:

In [ ]:
X_t = np.hstack([np.ones_like(google_time_step_t), label_time_step_t])
X_t

In [ ]:
# Computes the vector x that approximately solves the equation a @ x = b
θ_t, resid, _, _ = np.linalg.lstsq(X_t, google_time_step_t, rcond=-1) # set rcond to machine precision

In [ ]:
θ_t

Now let's predict four values of Google stock on this date:

In [ ]:
X_t @ θ_t 

In [ ]:
# residuals
X_t @ θ_t  - google_time_step_t

Now let's do loop over multiple time steps

In [ ]:
len_t = label_df.shape[0]
model_coeffs = np.zeros( ( len_t, len(labels)+1))

In [ ]:
for t in range(len_t):
    label_time_step_t = label_df.iloc[t].values.reshape(4, len(labels))
    google_time_step_t = google_df.iloc[t].values.reshape(4, 1)
    X_i = np.hstack([np.ones_like(google_time_step_t), label_time_step_t])
    coeffs, resid, _, _ = np.linalg.lstsq(X_i, google_time_step_t, rcond=-1)
    model_coeffs[t,:] = coeffs.T

n_model_parameters = model_coeffs.shape[1]

In [ ]:
plt.plot(label_df.index[0:50], model_coeffs[0:50, 1],label="CMG θ_1" )
plt.plot(label_df.index[0:50], model_coeffs[0:50, 2], label="NKE θ_2")
plt.hlines([0], xmin=label_df.index[0], xmax=label_df.index[50], color='grey')
plt.ylim(-25,25)
plt.legend()

Now let's try doing the fitting at one timestep by maximizing the likelihood

In [ ]:
def generate_google_at_one_timestep(θ, label_data):
    # calulate the target value given the labels
    X_t = np.hstack([1, label_data])
    return X_t @ θ

    
def neg_likelihood(θ_with_sigma, observed, label_data):

    # generate the (negative) log likelihood, given the labels and the observed data
    # and the model parameters  
    θ = θ_with_sigma[0:-1] # model parameters
    sigma_2 = θ_with_sigma[-1] # sigma
    
    model_value = generate_google_at_one_timestep(θ, label_data)
    
    term1 = (-1/2)*(model_value - observed)**2/sigma_2
    term2 = np.log(sigma_2)
    return -1*(term1 + term2)

In [ ]:
# 
t0 = 0
cmg_ = 60.790001
nke_ = 75.866013
ggl_ = 191.267093
θ = model_coeffs[t0,:].reshape(1, n_model_parameters).T


label_data = np.array([cmg_, nke_]).reshape(2)
generate_google_at_one_timestep(θ, label_data)

sigma_2 = 0.01 # assume constant noise -- doesn't have to be the case
θ_with_sigma = θ.reshape(3).tolist() + [sigma_2] # merge parameters and noise into a single list

neg_likelihood(θ_with_sigma, ggl_, label_data)
θ

In [ ]:
bnds = ((-1000, 1000), (-1000, 1000), (-1000, 1000), (0.0001, 1000000))
result = minimize(neg_likelihood, x0=(-120, 10, -5, 0.1), 
                  args=(ggl_,label_data), bounds=bnds, method='SLSQP') # Sequential Least Squares Programming (SLSQP)

In [ ]:
result.x

In [ ]:
θ

In [ ]:
%load_ext watermark

In [ ]:
%watermark -v -p numpy,pandas,yfinance,scipy,matplotlib  -a "J Bloom & D Weisz" -d -t